In [1]:
import os

In [2]:
%pwd

'd:\\Major\\TextSummarizer-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Major\\TextSummarizer-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config

In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch
import pandas as pd
from evaluate import load as load_metric

# Load the ROUGE metric (example)
rouge_metric = load_metric("rouge")


In [15]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk
from evaluate import load as load_metric
import torch
import pandas as pd
from tqdm import tqdm

class ModelEvaluation:
    def __init__(self, config):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split the dataset into smaller batches that we can process simultaneously."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(
        self, dataset, metric, model, tokenizer, 
        batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu", 
        column_text="article", 
        column_summary="highlights"
    ):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)
        ):
            inputs = tokenizer(
                article_batch, max_length=1024, truncation=True, 
                padding="max_length", return_tensors="pt"
            )
            
            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device), 
                length_penalty=0.8, num_beams=8, max_length=128
            )
            
            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True) 
                for s in summaries
            ]      

            metric.add_batch(predictions=decoded_summaries, references=target_batch)
            
        # Compute and return the metric scores.
        score = metric.compute()
        return score

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
       
        # Load data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_metric = load_metric("rouge")
        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
  
        # Evaluate on a subset of the test set for demonstration
        test_data = dataset_samsum_pt['test'].select(range(10))

        score = self.calculate_metric_on_test_ds(
            test_data, rouge_metric, model, tokenizer, 
            batch_size=2, column_text="dialogue", column_summary="summary"
        )

        # Adapt to the new structure of score
        rouge_dict = {rn: score[rn] for rn in rouge_names}
        
        # Save to CSV
        df = pd.DataFrame([rouge_dict])
        df.to_csv(self.config.metric_file_name, index=False)


In [16]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2024-12-13 04:29:52,555: INFO: common: yaml file: config\config.yaml loaded successfully]


[2024-12-13 04:29:52,568: INFO: common: yaml file: params.yaml loaded successfully]
[2024-12-13 04:29:52,577: INFO: common: created directory at: artifacts]
[2024-12-13 04:29:52,581: INFO: common: created directory at: artifacts/model_evaluation]


100%|██████████| 5/5 [06:10<00:00, 74.07s/it]


[2024-12-13 04:36:10,438: INFO: rouge_scorer: Using default tokenizer.]
